In [32]:
%reload_ext autoreload
%autoreload 2

import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)

In [33]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
import json
from pathlib import Path

from analyse.advertising.e00_download.mediatree import (
    CachedMediatreeAPI,
    all_intervals_between,
)
from analyse.advertising.e02_finger_printer.meta_matcher import (
    MetaMatcherPipeline,
)
from analyse.advertising.e02_finger_printer.finger_printer import (
    print_report,
    plot_groups,
)

tz_paris = ZoneInfo("Europe/Paris")
channel = "tf1"
output_prefix = "META"

MONDAY_LAST_YEAR = datetime(2025, 3, 10, tzinfo=tz_paris)

# ── Seuil de distance : plus petit = groupes plus stricts
# Valeurs indicatives après diagnostic :
#   0.05  → très strict (presque identiques)
#   0.15  → modéré (valeur par défaut)
#   0.30  → souple (peut sur-grouper)
DISTANCE_THRESHOLD = 0.02

# Différence de durée maximale pour être candidat au matching (secondes)
DURATION_HARD_MAX_SEC = 0.4

# Poids des features : [duration, rms, centroid, zcr]
# Augmenter un poids rend le critère correspondant plus discriminant
FEATURE_WEIGHTS = [1.5, 1.0, 1.0, 1.5]

OUTPUT_FILE = f"working_data/{output_prefix}_report_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.json"
OUTPUT_PLOT = f"working_data/{output_prefix}_plot_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}.png"

In [34]:
INPUT_SEGMENTS_FOLDER = os.path.join(
    "../working_data", f"segments_{channel}_{MONDAY_LAST_YEAR.strftime('%Y%m%d')}"
)

CUSTOM_DATES = [
    (
        datetime(2025, 3, 10, 12, 0, tzinfo=tz_paris),
        datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris),
    ),
    (
        datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris),
        datetime(2025, 3, 10, 13, 0, tzinfo=tz_paris),
    ),
    (
        datetime(2025, 3, 11, 12, 0, tzinfo=tz_paris),
        datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
    ),
    (
        datetime(2025, 3, 11, 12, 30, tzinfo=tz_paris),
        datetime(2025, 3, 11, 13, 0, tzinfo=tz_paris),
    ),
]
# CUSTOM_DATES = all_intervals_between(
#     start=MONDAY_LAST_YEAR,
#     end=MONDAY_LAST_YEAR + timedelta(days=7),
#     interval_length=timedelta(minutes=30),
# )

# Sources : (label_audio, chemin_json)
# Pas besoin de télécharger l'audio — on travaille uniquement sur les JSON
sources = []
for start_date, end_date in CUSTOM_DATES:
    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    if Path(segment_file).exists():
        sources.append((f"{channel}_{start_date.strftime('%Y%m%d_%H%M%S')}", segment_file))
    else:
        print(f"  [SKIP] fichier absent : {segment_file}")

print(f"{len(sources)} fichiers de segments chargés")

4 fichiers de segments chargés


## Diagnostic (optionnel)
Lance cette cellule pour explorer la distribution des distances et calibrer `DISTANCE_THRESHOLD`.

In [ ]:
if False:  # Mettre True pour lancer le diagnostic
    diag_pipeline = MetaMatcherPipeline(
        sources=sources,
        distance_threshold=DISTANCE_THRESHOLD,
        duration_hard_max_sec=DURATION_HARD_MAX_SEC,
        feature_weights=FEATURE_WEIGHTS,
    )
    segs = diag_pipeline.load_segments()
    diag_pipeline.normalizer.fit_transform(segs)
    diag_pipeline.diagnose(segs, top_n=20)

   309 segments  ←  ../working_data/segments_tf1_20250310/2025-03-10_12-00-00.json
   259 segments  ←  ../working_data/segments_tf1_20250310/2025-03-10_12-30-00.json
   305 segments  ←  ../working_data/segments_tf1_20250310/2025-03-11_12-00-00.json
   243 segments  ←  ../working_data/segments_tf1_20250310/2025-03-11_12-30-00.json

  Total : 1116 segments  (1116 avec features acoustiques, 0 sans)

────────────────────────────────────────────────────────────
  DIAGNOSTIC META-MATCHER
────────────────────────────────────────────────────────────

  Top 20 paires les plus proches :
   Seg A   Seg B    Distance   Durée A   Durée B
  ────────────────────────────────────────────────────
     316     851      0.0006    42.84s    42.82s
     499     960      0.0011     1.07s     1.09s
     919    1092      0.0022     1.21s     1.25s
     919     960      0.0055     1.21s     1.09s
     317     852      0.0064    13.21s    13.17s
     499     919      0.0066     1.07s     1.21s
     983     998  

## Run — grouping par métaparamètres

In [36]:
pipeline = MetaMatcherPipeline(
    sources=sources,
    distance_threshold=DISTANCE_THRESHOLD,
    duration_hard_max_sec=DURATION_HARD_MAX_SEC,
    feature_weights=FEATURE_WEIGHTS,
)

report, segments, groups = pipeline.run()
print_report(report)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(f"  Rapport JSON : {Path(OUTPUT_FILE).absolute()}")

total = max(s.end_sec for s in segments) if segments else 3600
plot_groups(report, total, OUTPUT_PLOT)
print(f"  Graphique    : {Path(OUTPUT_PLOT).absolute()}")


[MetaMatcher] Chargement des segments JSON...
   309 segments  ←  ../working_data/segments_tf1_20250310/2025-03-10_12-00-00.json
   259 segments  ←  ../working_data/segments_tf1_20250310/2025-03-10_12-30-00.json
   305 segments  ←  ../working_data/segments_tf1_20250310/2025-03-11_12-00-00.json
   243 segments  ←  ../working_data/segments_tf1_20250310/2025-03-11_12-30-00.json

  Total : 1116 segments  (1116 avec features acoustiques, 0 sans)

[MetaMatcher] Normalisation des features...

  Features — plages effectives après normalisation :
    duration     min=1.068  range=31.79
    rms          min=0.007698  range=0.04052
    centroid     min=1295  range=2140
    zcr          min=0.04918  range=0.1712

[MetaMatcher] 1116 segments — comparaison par features...


Paires: 100%|██████████| 1116/1116 [00:00<00:00, 8073.30it/s]

  64037 paires candidates (filtre durée ±0.4s)
  25 paires similaires (distance < 0.02)



══════════════════════════════════════════════════════════════════════
  RAPPORT DE RÉPÉTITIONS
══════════════════════════════════════════════════════════════════════
  Segments analysés : 1116
  Groupes détectés  : 1095

  Classification             Groupes   Occurrences tot
  ────────────────────────────────────────────────────
  SEGMENT_UNIQUE                1077              1077
  CONTENU_REPETE                  18                39

  TOP 15 GROUPES LES PLUS FRÉQUENTS :
  Groupe    Occurrences   Durée moy  Classification
  ────────────────────────────────────────────────────────────
  G1092               4       1.2s  CONTENU_REPETE
    ↳ [2025-03-10 11:51:19 UTC]  1.1s
    ↳ [2025-03-11 11:37:01 UTC]  1.2s
    ↳ [2025-03-11 11:38:59 UTC]  1.1s
    ↳ ... +1 autres occurrences
  G998                3       1.2s  CONTENU_REPETE
    ↳ [2025-03-10 11:44:09 UTC]  1.4s
    ↳ [2025-03-11 11:40:40 UTC]  1.0s
    ↳ [2025-03-11 11:41:16 UTC]  1.2s
  G1110               2       3.4s  CONTE

## Visualisation dans le player (optionnel)
Génère les pages HTML avec les groupes annotés.

In [37]:
import httpx
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.tools.basic_elements.segments import load_segment_groups_from_json
from analyse.advertising.e01_rupture_detection.rupture_detector import RuptureDetector

api = CachedMediatreeAPI()
TESTIMONY_CHANNEL = channel.upper()
detector = RuptureDetector()  # paramètres par défaut pour les métadonnées uniquement

for start_date, end_date in CUSTOM_DATES:
    segment_file = os.path.join(
        INPUT_SEGMENTS_FOLDER,
        f"{start_date.strftime('%Y-%m-%d_%H-%M-%S')}.json",
    )
    if not Path(segment_file).exists():
        continue

    annotations = get_testimony_data(
        channel=TESTIMONY_CHANNEL,
        from_date=start_date,
        to_date=end_date,
        source_file="export.csv",
    )

    async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client:
        witness_file = await api.api.get_single_export_url(
            client, channel, start_date, end_date, media_format="mp4"
        )

    segs = load_segment_groups_from_json(segment_file)
    output_path = f"working_data/{output_prefix}/result_{channel}_{start_date.strftime('%Y%m%d_%H%M%S')}.html"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    generate_player(
        media_input=witness_file,
        segments=[s.to_alternary_dict() for s in segs],
        annotations=format_annotation(annotations, from_date=start_date),
        output_path=output_path,
        start_epoch=int(start_date.timestamp()),
        params=detector.get_params(),
        groups=report["groups"],
    )

[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-10T11-00-00Z_2025-03-10T11-30-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  309 segments
[3/3] Annotations fournies en entrée
  1 annotations

Génération du rapport HTML : working_data/META/result_tf1_20250310_120000.html...

✓  Rapport généré  : /root/Workspace/quotaclimat/analyse/advertising/e02_finger_printer/working_data/META/result_tf1_20250310_120000.html
   Taille HTML     : 0.5 Mo (léger, média chargé à la volée)
   URL média       : https://vod.mediatree.fr/assets/tf1_2025-03-10T11-00-00Z_2025-03-10T11-30-00Z.mp4
   Segments        : 309
   Annotations CSV : 1

   → Ouvrez dans Firefox ou Chrome pour l'analyse.

[1/3] URL média détectée : https://vod.mediatree.fr/assets/tf1_2025-03-10T11-30-00Z_2025-03-10T12-00-00Z.mp4
  Type     : vidéo
  Le fichier ne sera PAS embarqué dans le HTML.
[2/3] Segments fournis en entrée
  259 segments
[3/3] Annota